# 6. DESCARGA E INTERPRETACIÓN DE IMAGENES LANDSAT 8 Y SENTINEL 1 #

- Descarga de imagenes Landsat 8 con índices
- Inferencia de plataforma S1A / S1B
- Selección escenas:
    - Misms fecha
    - Ventana post-incendio 

<span style="color:red"> Se debe buscar alguna mascara para pluma de humo derivado de los incendios. Podría ser la causante de gran parte de la imagen no cuente con datos </span>

## 6.1  MOSAICOS MENSUALES DE LANDSAT 8 PARA EL PERIODO DE ANALISIS (2014 - 2024) ##

Genera y exportar composites mensuales de Landsat 8 Collection 2 Tier 1 Level-2 entre 2014 y 2024 para un AOI definido por shapefile. En términos operativos, el notebook: Lee un AOI desde shapefile y lo transforma a ee.Geometry en WGS84 (EPSG:4326), aplicando correcciones de geometría (validación, buffer(0), unión). Construye una colección mensual (ImageCollection) filtrada por fecha y AOI.

Aplica: enmascaramiento QA (nubes/sombras y saturación radiométrica), escalado a reflectancia para bandas SR, 
una máscara topográfica vía hillShadow (control de sombra por relieve), y calcula índices espectrales NDVI, EVI, NBR y NDWI. Calcula un composite mediana por mes. Añade una banda IMG_COUNT con el número de imágenes del mes.

Exporta cada composite a Google Drive como GeoTIFF (30 m), creando 11×12 = 132 tareas.

###  6.1.1 Parte 1 ###

Autenticación e inicialización de GEE; lectura y preparación del área de interes, reproyectar y validar geometria. Definición de funciones e índices. Se realiza mascara topografica usando el SRTM y geometría solar, devolviendo imagenes con bandas corregidas. Se emplean el índice NDVI, EVI, NBR y NDWI. Se define las fechas para el mosaico, filtrandose por fecha y área de interes. Se obtiene la mediana del mes y el número de iamgenes empleadas para el mosaico. Finalmente se exporta a google drive con resolcuión de 30 metros y se genera un reporte final por tareas. Los datos aquí suministrados, de momento, se manejan es sistema de coordenadas EPSG = 4326. 


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import ee
from src.gee_aoi import load_aoi_ee_geometry
from src.gee_export import setup_logger, run_monthly_exports

# 1) Auth solo cuando toque
ee.Authenticate()
ee.Initialize()

ruta_aoi_shp = r"D:\Maestria_Geomatica\Semestre_IV\Tesis_Python\Datos\SHP\AOI_25_09_19.shp"
aoi = load_aoi_ee_geometry(ruta_aoi_shp, target_epsg=4326)

# params
year_start, year_end = 2014, 2024
folder_drive = "Imagenes_GEE"
scale = 30
dry_run = False  # True para probar sin lanzar tareas

log_path = os.path.join(os.getcwd(), "logs", "gee_exports.log")
logger = setup_logger(log_path)

tasks = run_monthly_exports(
    aoi=aoi,
    year_start=year_start,
    year_end=year_end,
    folder=folder_drive,
    scale=scale,
    logger=logger,
    dry_run=dry_run
)

print("Total tareas:", len(tasks))
tasks[:5]

Total tareas: 132


[TaskInfo(year=2014, month=1, desc='L8_2014_01'),
 TaskInfo(year=2014, month=2, desc='L8_2014_02'),
 TaskInfo(year=2014, month=3, desc='L8_2014_03'),
 TaskInfo(year=2014, month=4, desc='L8_2014_04'),
 TaskInfo(year=2014, month=5, desc='L8_2014_05')]

## 6.2 Preprocesamiento Imagenes Sentinel 1 GRD ##

Consulta la colección Sentinel-1 GRD (COPERNICUS/S1_GRD) para un año específico (YEAR_TO_RUN) y un sentido orbital (ASCENDING o DESCENDING). Genera dos productos complementarios: Un CSV índice (metadatos por escena) exportado a Google Drive; exportación de imágenes procesadas por lotes, con preprocesamiento SAR (máscara de borde, normalización gamma0, ratio VV/VH, filtrado speckle multitemporal y diferencia VV respecto a una referencia). Se descargaron 458 imagenes en orbita ascendente entre diciembre de 2014 y diciembre de 2021. No se encontraron imagenes para meses anteriores a diciembre de 2014 e incosistencia de imagenes mensuales para 2015 y 2016. No se encontraron imagenes de orbita ascendente desde el año 2022 al 2023.

<font color="red">
    
<font color="red"> Dudas :</font>

El filtrado de speckle basado en imágenes previas podría sesgar la caracterización del área quemada. Aunque en el modelo Random Forest se seleccionen imágenes de un mes posterior al evento (por ejemplo, del 04 de febrero para un incendio ocurrido el 04 de enero), si el proceso de reducción de ruido utiliza un histórico anterior a esa fecha, la  de retrodispersión del incendio podría suavizarse o alterarse con información del estado previo de la vegetación.

Todo esta en dB por lo que VV_Difference y VV_ratio no darían bien

Se deberia calcular en lineal, conserva el signo físico, y se pasa a “dB con signo” con sign(diff_lin)*10*log10(max(|diff_lin|, epsilon)) no sé si sería correcto de esta forma
</font>

### 6.2.1 Parte 1 ###

Inicializa GEE; construye el área de interes como ee.Geometry desde shapefile y define parámetros globales (año, órbita, tamaño de lote); define funciones de preprocesamiento SAR con orbita ascendente. Construcción de la colección Sentinel 1, definiendose la selección de escenas Sentienl 1 que intersectan el área de interes, estan dentro del año, son IW GRD, cuentan con polarizaciones VV y VH y para la orbita previamente definida (ascendente). Se realiza Speckle multitemporal basado en imágenes previas. Para cada imagen, se construye una mediana de las N_PREV_IMAGES inmediatamente anteriores (en tiempo), y luego suavizar espacialmente con un kernel boxcar. Se calcula Banda de diferencia VV respecto a referencia, permitiendo realizar cambios respecto a una condición base del año. Se exporta CSV con los metadatos de las imagenes

<font color="red">
    
El siguiente codigo no genera vv/vh ratio ni vv_difference aún. El speackel deberia realizar primero, antes de la normalización 
</font>

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import json
import logging
from pathlib import Path
import sys
import ee

# =========================
# 0) PROJECT ROOT (una sola verdad)
# =========================
PROJECT_DIR = Path.cwd().resolve()

# Validaciones rápidas (si algo falla aquí, el problema es de ruta real)
print("PROJECT_DIR:", PROJECT_DIR)
print("PROJECT_DIR exists:", PROJECT_DIR.exists())

CFG_PATH = PROJECT_DIR / "configs" / "s1_default.json"
LOG_PATH = PROJECT_DIR / "logs" / "run.log"

print("PROJECT_DIR:", PROJECT_DIR)
print("CFG_PATH:", CFG_PATH)
print("CFG exists:", CFG_PATH.exists())

# Si esto imprime False, el archivo NO está ahí (o la letra D: no es la que crees)
# y por eso "sigue el mismo error".
if not CFG_PATH.exists():
    raise FileNotFoundError(f"No existe el config en: {CFG_PATH}")

# =========================
# 1) Logging reproducible (en ruta ABSOLUTA)
# =========================
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    filename=str(LOG_PATH),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logging.info("=== START RUN ===")

# =========================
# 2) Asegurar imports de src/
# =========================
# Necesitas que PROJECT_DIR esté en sys.path para 'import src...'
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# =========================
# 3) Earth Engine init
# =========================
ee.Authenticate()
ee.Initialize()

# =========================
# 4) Cargar config
# =========================
cfg = json.loads(CFG_PATH.read_text(encoding="utf-8"))
print("Config cargado OK")

# =========================
# 5) AOI desde shapefile
# =========================
from src.aoi import aoi_shp_to_ee_geometry

AOI_PATH = r"D:\Maestria_Geomatica\Semestre_IV\Tesis_Python\Datos\SHP\AOI_25_09_19.shp"
aoi = aoi_shp_to_ee_geometry(AOI_PATH)

# =========================
# 6) Pipeline (IMPORTA build_pipeline)
# =========================
from src.pipeline_sentinel1 import build_pipeline  # <-- AJUSTA si tu módulo se llama distinto

col_final = build_pipeline(aoi, cfg)

# =========================
# 7) Validación (getInfo SOLO aquí)
# =========================
bands = ee.Image(col_final.first()).bandNames().getInfo()
n = col_final.size().getInfo()

print("Bandas:", bands)
print("N imágenes:", n)

logging.info(f"bands={bands}")
logging.info(f"n_images={n}")

PROJECT_DIR: C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20
PROJECT_DIR exists: True
PROJECT_DIR: C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20
CFG_PATH: C:\Users\Equipo\Tesis\SemestreIV\Objetivo1\Objetivo1_26_02_20\configs\s1_default.json
CFG exists: True
Config cargado OK
Bandas: ['VV', 'VH', 'angle', 'VVVH_ratio', 'VV_Difference']
N imágenes: 114
